# Lab 10 — Web Scraping a Static Public Page
**Data Acquisition Track** · Intermediate · ~45 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Parse an HTML table with html.parser (BeautifulSoup optional)
2. Normalise rows to dicts and handle missing cells
3. Export scraped.csv with a row-count assertion
4. Practice polite scraping rules and robots.txt awareness

## Datasets (this folder)
- `fixture.html` — **upload** in Colab (or keep next to the notebook locally)

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-01-foundations-python-math-data/labs/lab-10-web-scraping-static-page/lab-10-web-scraping-static-page.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0** first — it downloads `dataset.zip` with wget, unzips it, and every code cell below reads those unzipped files.
3. **Runtime → Run all** (GPU not required for Course 1).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 (`wget dataset.zip` + `unzip`) → `Runtime → Run all`.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`fixture.html`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — dataset first: wget dataset.zip + unzip (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-10-web-scraping-static-page"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-01-foundations-python-math-data/bundles/lab-10-web-scraping-static-page/dataset.zip"
NEED = ["fixture.html"]  # unzipped files used by the code below

def _have_files():
    return all(os.path.exists(f) for f in NEED)

def _wget_zip(url, dest):
    # shell equivalent: !wget -q <url> -O dataset.zip
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", url, "-O", dest], check=True)
    else:  # plain Python without wget: stdlib fallback
        urllib.request.urlretrieve(url, dest)

if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    _wget_zip(DATASET_ZIP_URL, "dataset.zip")
    # shell equivalent: !unzip -o -q dataset.zip
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))


## Data Acquisition Track: HTML Tables → CSV (Offline Fixture First)

> **Scenario:** Extract a product table from HTML with `urllib` + `html.parser` (or BeautifulSoup if available), handle missing cells, and save `scraped.csv`. A local `fixture.html` ships with this lab so **everything runs without network**. An optional live target (`quotes.toscrape.com`) is included for when you do have egress.
>
> **You will learn:** HTML structure, table parsing, polite scraping (User-Agent, delay), robots.txt awareness, anti-patterns warning.
> **Time:** ~45 minutes. **Level:** Intermediate. **Needs:** Python 3.8+ (BeautifulSoup optional). **Env:** 🟢 Colab only.

### Scraping mental map

| HTML piece | Python view | Extract how |
|---|---|---|
| `<table>` | container | `HTMLParser` handle_starttag |
| `<tr>` | row | accumulate cells |
| `<td>` | cell text | `handle_data` between tags |
| empty `<td>` | missing value | `""` → flag / impute |
| CSS selector | BeautifulSoup | `soup.select("table#products tr")` |

> **Politeness rules:** identify yourself with a `User-Agent`, cache results, don’t hammer servers, and respect `robots.txt` + ToS. Scraping behind logins or against explicit disallow paths is not okay.

---

### 1. Load local fixture (offline path — always works)

In [ ]:
import csv, os, re, time
from html.parser import HTMLParser

FIXTURE = "fixture.html"
LIVE_URL = "https://quotes.toscrape.com/"  # optional, public practice site

def read_local(path=FIXTURE):
    with open(path, encoding="utf-8") as f:
        return f.read()

html = read_local()
print("fixture bytes:", len(html))
assert "<table" in html and "Widget A" in html


If the fixture is missing, the notebook writes a minimal one:

In [ ]:
if not os.path.exists(FIXTURE):
    open(FIXTURE, "w", encoding="utf-8").write(
        "<table id='products'><tr><th>ID</th><th>Name</th><th>Price</th><th>Stock</th></tr>"
        "<tr><td>1</td><td>Widget A</td><td>9.99</td><td>120</td></tr></table>"
    )


---

### 2. Stdlib parser: table rows → list of lists

In [ ]:
class TableParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.in_td = False
        self.in_tr = False
        self.cell = ""
        self.row = []
        self.rows = []
        self.in_header = False

    def handle_starttag(self, tag, attrs):
        if tag == "tr":
            self.in_tr = True
            self.row = []
        elif tag in ("td", "th"):
            self.in_td = True
            self.cell = ""
        if tag == "th":
            self.in_header = True

    def handle_endtag(self, tag):
        if tag in ("td", "th"):
            self.in_td = False
            self.row.append(self.cell.strip())
        elif tag == "tr":
            self.in_tr = False
            if self.row:
                self.rows.append(self.row)
            self.in_header = False

    def handle_data(self, data):
        if self.in_td:
            self.cell += data

def parse_table(html_text):
    p = TableParser()
    p.feed(html_text)
    if not p.rows:
        return [], []
    header, *body = p.rows
    return header, body

header, body = parse_table(html)
print("header:", header)
# ['ID', 'Name', 'Price', 'Stock']
print("n data rows:", len(body))
# 5
print("first row:", body[0])
# ['1', 'Widget A', '9.99', '120']


---

### 3. Rows → dicts; handle missing cells

Fixture intentionally has a **blank Price** on Gadget C:

In [ ]:
def rows_to_dicts(header, body):
    out = []
    for r in body:
        # pad short rows
        r = list(r) + [""] * (len(header) - len(r))
        rec = dict(zip(header, r))
        # normalise numerics with fallbacks
        rec["Price"] = float(rec["Price"]) if rec["Price"] not in ("", None) else None
        rec["Stock"] = int(rec["Stock"]) if str(rec["Stock"]).strip().isdigit() else None
        out.append(rec)
    return out

records = rows_to_dicts(header, body)
for rec in records:
    print(rec)
# {'ID': 1, 'Name': 'Widget A', 'Price': 9.99, 'Stock': 120}
# ...
# {'ID': 3, 'Name': 'Gadget C', 'Price': None, 'Stock': 7}   # missing price

missing_price = [r for r in records if r["Price"] is None]
print("missing prices:", len(missing_price))  # 1


> Always assert row counts and required keys after parsing — silent empty scrapes are the #1 bug.

---

### 4. Write `scraped.csv` + row-count assert

In [ ]:
out_path = "scraped.csv"
fields = ["ID", "Name", "Price", "Stock"]
with open(out_path, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    for r in records:
        w.writerow({
            "ID": r["ID"],
            "Name": r["Name"],
            "Price": "" if r["Price"] is None else f"{r['Price']:.2f}",
            "Stock": "" if r["Stock"] is None else r["Stock"],
        })

with open(out_path, encoding="utf-8") as f:
    n = sum(1 for _ in f)
assert n == 1 + len(records), f"expected {1+len(records)} lines, got {n}"
print(f"wrote {out_path}: {len(records)} data rows, {n} lines total")
# wrote scraped.csv: 5 data rows, 6 lines total


---

### 5. Optional: BeautifulSoup (if installed)

In [ ]:
try:
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(html, "html.parser")
    trs = soup.select("table#products tbody tr")
    print("bs4 rows:", len(trs))  # 5
    bs_rows = []
    for tr in trs:
        tds = [td.get_text(strip=True) for td in tr.find_all("td")]
        bs_rows.append(tds)
    print(bs_rows[0])  # ['1', 'Widget A', '9.99', '120']
except ImportError:
    print("BeautifulSoup not installed — stdlib parser above is sufficient")


---

### 6. Optional live fetch (network only)

In [ ]:
def fetch_live(url, delay=1.0, timeout=10):
    import urllib.request
    time.sleep(delay)  # be polite
    req = urllib.request.Request(url, headers={"User-Agent": "ailab-lab10/1.0 (educational)"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.read().decode("utf-8", errors="replace")

# Live scrape is optional; lab verification uses fixture.html only.
# if os.environ.get("AILAB_ALLOW_NET"):
#     page = fetch_live(LIVE_URL)
#     print("live bytes:", len(page))
print("offline mode: skipping live fetch (set AILAB_ALLOW_NET=1 to enable)")


> Check `https://quotes.toscrape.com/robots.txt` before scraping any live site. Practice domains exist for a reason.

---

## Exercises (do these!)

### Exercise 1 — Parse table rows to dicts
Using `TableParser` + `rows_to_dicts`, print the number of rows and the full list of `Name` values.
*Expected: 5 rows · names: Widget A, Widget B, Gadget C, Gadget D, Doohickey E.*

**Follow-up:** How many units are in stock across all products? Check: 472.

<details>
<summary>Hint</summary>

`header, body = parse_table(html)` then `rows_to_dicts(header, body)`; collect `r["Name"]`.
</details>

### Exercise 2 — Handle missing cells
Which product has an empty `Price`? How many missing prices? What Python value do you store?
*Expected: Gadget C · 1 missing · store `None` (not `0` — zero price is a real value).*

**Follow-up:** What is the mean price over priced products? Check: about 12.685.

<details>
<summary>Hint</summary>

`[r for r in records if r["Price"] is None]`.
</details>

### Exercise 3 — Write CSV + row-count assert
Write `scraped.csv` and assert `lines == 1 + len(records)`. Print the assertion message.
*Expected: 5 data rows → 6 lines; assert passes.*

**Follow-up:** Prove the CSV header. Check: ID,Name,Price,Stock.

<details>
<summary>Hint</summary>

`sum(1 for _ in open(path))` includes the header.
</details>

---

## Solutions

In [ ]:
# --- Solution 1 ---
header, body = parse_table(read_local())
records = rows_to_dicts(header, body)
print(len(records), [r["Name"] for r in records])
# 5 ['Widget A', 'Widget B', 'Gadget C', 'Gadget D', 'Doohickey E']

# --- Solution 2 ---
missing = [r["Name"] for r in records if r["Price"] is None]
print(missing, len(missing), "-> Price stored as None")
# ['Gadget C'] 1 -> Price stored as None

# --- Solution 3 ---
# (Section 4 already writes scraped.csv)
with open("scraped.csv", encoding="utf-8") as f:
    lines = sum(1 for _ in f)
assert lines == 1 + len(records)
print("OK", lines, "lines for", len(records), "records")
# OK 6 lines for 5 records

# --- Follow-up 1 ---
units = sum(r["Stock"] for r in records)
print(units)  # 472
assert units == 472

# --- Follow-up 2 ---
priced = [r["Price"] for r in records if r["Price"] is not None]
mp = sum(priced) / len(priced)
print(round(mp, 3))  # ~12.685
assert abs(mp - 12.685) < 1e-9

# --- Follow-up 3 ---
with open("scraped.csv", encoding="utf-8") as f:
    header_line = f.readline().strip()
print(header_line)
assert header_line == "ID,Name,Price,Stock"


### What to learn next
- CSS selectors with BeautifulSoup: `soup.select_one("td.price")`.
- APIs behind the page (view Network tab) often beat HTML scraping.
- `scrapy` / `playwright` for JS-heavy sites — and legal review first.
- Cheat sheet: fixture-first → parse → normalise missings → assert counts → export → only then go live.

*Files in this folder: `fixture.html` (offline table) · output `scraped.csv`.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
